# Amazon Bedrock AgentCore Observability - Agent attribute redaction

## Overview

In this tutorial, we will also demonstrate how to redact Agent prompts at source using Opentelemetry for AgentCore Observability.

## Prerequisites

To execute this tutorial you will need:
* Python 3.10+
* AWS credentials
* Amazon Bedrock AgentCore SDK
* Strands Agents
* Docker running
* Amazon CloudWatch Access
* Enable [transaction search](https://docs.aws.amazon.com/AmazonCloudWatch/latest/monitoring/Enable-TransactionSearch.html) on Amazon CloudWatch. 


In [ ]:
!uv add -r requirements-dev.txt --quiet
!uv add -r requirements.txt --quiet

## Preparing your agent for deployment on AgentCore Runtime

Let's now deploy our agents to AgentCore Runtime. To do so we need to:
* Import the Runtime App with `from bedrock_agentcore.runtime import BedrockAgentCoreApp`
* Initialize the App in our code with `app = BedrockAgentCoreApp()`
* Decorate the invocation function with the `@app.entrypoint` decorator
* Let AgentCoreRuntime control the running of the agent with `app.run()`

### Strands Agents with Amazon Bedrock model
Let's start with our Strands Agent using Amazon Bedrock model. All the others will work exactly the same.

In [ ]:
%%writefile agent.py
"""Sample Strands agent with attribute redaction"""

# pylint:disable=logging-fstring-interpolation

import logging
from strands import Agent, tool
from strands.models import BedrockModel
from strands_tools import calculator  # Import the calculator tool
from bedrock_agentcore.runtime import BedrockAgentCoreApp

from opentelemetry import baggage, context as trace_context
from opentelemetry.sdk.trace import SpanProcessor
from opentelemetry.trace import get_tracer_provider
from opentelemetry.processor.baggage import BaggageSpanProcessor, ALLOW_ALL_BAGGAGE_KEYS

# Set up logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)
logger.setLevel("DEBUG")

# Init App service
MODEL_ID = "us.anthropic.claude-sonnet-4-5-20250929-v1:0"
SYSTEM_PROMPT = """
You're a helpful assistant. You can do simple math calculation, and tell the weather.
"""

app = BedrockAgentCoreApp()


class SensitiveDataRedactor(SpanProcessor):
    """Custom OpenTelemetry SpanProcessor to redact attributes"""
    SENSITIVE_ATTRS = [
        "llm.prompts", "gen_ai.input.messages", "llm.completions", "user.email"
    ]

    def on_end(self, span):
        """Override on_end to redact select attributes"""
        if span.attributes:
            for attr in self.SENSITIVE_ATTRS:
                if attr in span.attributes:
                    span._attributes[attr] = "[REDACTED]"


tracer_provider = get_tracer_provider()
tracer_provider.add_span_processor(BaggageSpanProcessor(ALLOW_ALL_BAGGAGE_KEYS))
tracer_provider.add_span_processor(SensitiveDataRedactor())


@tool
def weather():
    """ Get weather """
    # Dummy implementation
    return "sunny"


def set_baggage():
    """Session ID OpenTelemetry baggage for distributed trace correlation"""
    ctx = baggage.set_baggage("tenant.id", "demo_user")
    ctx = baggage.set_baggage("user.email", "demo_user@anycompany.com", context=ctx)
    token = trace_context.attach(ctx)
    print("Baggage context attached")
    return token


def init_agent():
    """Creates an agent"""
    model = BedrockModel(
        model_id=MODEL_ID,
    )
    set_baggage()
    agent = Agent(
        model=model,
        tools=[calculator, weather],
        system_prompt=SYSTEM_PROMPT
    )
    return agent


@app.entrypoint
def strands_agent_bedrock(payload):
    """
    Invoke the agent with a payload
    """
    user_input = payload.get("prompt")
    print("User input:", user_input)

    agent = init_agent()
    response = agent(user_input)
    return response.message['content'][0]['text']


if __name__ == "__main__":
    app.run()


## Deploying the agent to AgentCore Runtime

The `CreateAgentRuntime` operation supports comprehensive configuration options, letting you specify container images, environment variables and encryption settings. You can also configure protocol settings (HTTP, MCP) and authorization mechanisms to control how your clients communicate with the agent. 

**Note:** Operations best practice is to package code as container and push to ECR using CI/CD pipelines and IaC

In this tutorial can will the Amazon Bedrock AgentCode Python SDK to easily package your artifacts and deploy them to AgentCore runtime.

### Configure AgentCore Runtime deployment

Next we will use our starter toolkit to configure the AgentCore Runtime deployment with an entrypoint, the execution role we just created and a requirements file. We will also configure the starter kit to auto create the Amazon ECR repository on launch.

During the configure step, your docker file will be generated based on your application code. 

Please note that when using the `bedrock_agentcore_starter_toolkit` to configure your agent, it takes care of the opentelemetry instrumentation. 

When configuring for containerized environment (such as docker) add the following command, an example is given below:

`CMD ["opentelemetry-instrument", "python", "runtime_agent_main.py"]`



In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime
from boto3.session import Session
boto_session = Session()
region = boto_session.region_name

agentcore_runtime = Runtime()
agent_name = "obs_attrib_redact_demo"
response = agentcore_runtime.configure(
    entrypoint="agent.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    agent_name=agent_name,
    memory_mode='NO_MEMORY'
)
response

### Launching agent to AgentCore Runtime

Now that we've got a docker file, let's launch the agent to the AgentCore Runtime. This will create the Amazon ECR repository and the AgentCore Runtime

In [ ]:
launch_result = agentcore_runtime.launch(
    # env_vars={
    #     "OTEL_PYTHON_EXCLUDED_URLS": "*/invocations"
    # }
)
launch_result

### Checking for the AgentCore Runtime Status
Now that we've deployed the AgentCore Runtime, let's check for it's deployment status

In [ ]:
import time
status_response = agentcore_runtime.status()
status = status_response.endpoint['status']
end_status = ['READY', 'CREATE_FAILED', 'DELETE_FAILED', 'UPDATE_FAILED']
while status not in end_status:
    time.sleep(10)
    status_response = agentcore_runtime.status()
    status = status_response.endpoint['status']
    print(status)
status

### Invoking AgentCore Runtime

Finally, we can invoke our AgentCore Runtime with a payload

### Processing invocation results

We can now process our invocation results to include it in an application

In [ ]:
from IPython.display import Markdown, display
invoke_response = agentcore_runtime.invoke({"prompt": "How is the weather now?"})
response_text = invoke_response['response'][0]
display(Markdown(response_text))

In [ ]:
!agentcore invoke '{"prompt": "How is the weather now?"}'

## Cleanup (Optional)

Let's now clean up the AgentCore Runtime created

In [ ]:
!agentcore destroy --force --delete-ecr-repo --dry-run

# Congratulations!